# TrOCR run 3: mixed real + synthetic fine-tune

Fine-tunes the run-2 Polish model (`PiotrSty/trocr-pl-base`) on a mix of synthetic
print lines (`ocr-pl-lines`) and **real typewritten** EHRI Polish lines
(`ehri-pl-lines`). Validation uses the held-out EHRI dev document; the frozen
EHRI test documents and the printed `real-lines-v1` benchmark are only used for
final evaluation. Select GPU T4. Code, model and data revisions are pinned.

In [ ]:
import subprocess, sys, os
from pathlib import Path
CODE_REVISION = '1088d23d2fa4adc20090a56758c338bd1b91442f'
BASE_MODEL = 'PiotrSty/trocr-pl-base'
BASE_REVISION = 'fff0416a9ccd8786cbd6f12d4cf3147c07056b18'
SYN_REVISION = 'd881debb90045fd71ad8e25faeeafeb6adab6622'
EHRI_REVISION = '3003e8614b74a351e7d94aba4f1348368815fb70'
repo = Path('/kaggle/working/OCR_engine')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.6','peft==0.19.1','jiwer','pillow','accelerate'],check=True)

In [ ]:
import tarfile
import torch
assert torch.cuda.is_available(), 'GPU is required; select Kaggle GPU T4.'
print('GPU:', torch.cuda.get_device_name(0), 'torch:', torch.__version__)
probe = torch.ones((2, 2), device='cuda')
assert (probe @ probe).sum().item() == 8.0
torch.cuda.synchronize(); del probe
print('CUDA_PREFLIGHT_OK', flush=True)
from huggingface_hub import hf_hub_download
from training.protocol import pair_manifest

syn_archive = hf_hub_download('PiotrSty/ocr-pl-lines','ocr-pl-lines-v1.tar.gz',repo_type='dataset',revision=SYN_REVISION)
syn_root = Path('/kaggle/working/ocr-pl-lines-v1'); syn_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(syn_archive,'r:gz') as b: b.extractall(syn_root, filter='data')

ehri_archive = hf_hub_download('PiotrSty/ehri-pl-lines','ehri-pl-lines-v1.tar.gz',repo_type='dataset',revision=EHRI_REVISION)
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1'); ehri_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(ehri_archive,'r:gz') as b: b.extractall(ehri_root, filter='data')

print('synthetic train:', len(pair_manifest(syn_root/'train')), 'val:', len(pair_manifest(syn_root/'val')))
print('ehri train:', len(pair_manifest(ehri_root/'train')), 'dev:', len(pair_manifest(ehri_root/'dev')), 'test:', len(pair_manifest(ehri_root/'test')))

In [ ]:
# Fine-tune run-2 on mixed synthetic + real EHRI lines; validate on held-out EHRI dev.
output = '/kaggle/working/trocr-pl-run3'
subprocess.run([sys.executable,'-m','training.train_trocr_pl',
    '--train-dir',f'{syn_root}/train',f'{ehri_root}/train',
    '--val-dir',f'{ehri_root}/dev',
    '--base',BASE_MODEL,'--revision',BASE_REVISION,'--output',output,
    '--epochs','5','--batch-size','8','--lr','1e-4','--no-4bit'],check=True)
print(Path(output,'selection.json').read_text())
print(Path(output,'best_metrics.json').read_text())
# No upload_folder: review CER on held-out test before any promotion.

In [ ]:
# Final evaluation on frozen held-out sets — self-contained cell (safe to rerun alone).
import sys, subprocess
from pathlib import Path
sys.path.insert(0, '/kaggle/working/OCR_engine')
BASE_MODEL = 'PiotrSty/trocr-pl-base'
output = '/kaggle/working/trocr-pl-run3'
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1')
real_lines = '/kaggle/working/OCR_engine/benchmarks/real-lines-v1/pairs'
for name, model in [('run3', output), ('run2-base', BASE_MODEL)]:
    for split, data in [('ehri-test', f'{ehri_root}/test'), ('real-lines-v1', real_lines)]:
        print(f'=== {name} on {split} ===', flush=True)
        subprocess.run([sys.executable,'-m','training.evaluate','--data',data,
            '--model',model,'--device','cuda','--batch-size','16'], check=True)